# Course: Data Science Course
# Author: Sandro Camargo <sandrocamargo@unipampa.edu.br>
# Class 03 - Datasus


To open this code in your Google Colab environment, [click here](https://colab.research.google.com/github/Sandrocamargo/data-science/blob/master/cd03_Melanoma.ipynb).

# Projeto 1

**Descrição geral**

Neste projeto, o estudante deverá realizar um estudo de análise temporal utilizando uma base de dados real, com o objetivo de investigar a evolução de um fenômeno ao longo do tempo.

A base de dados deve ser original e individual, não podendo coincidir com a de outros colegas.

**Contexto aplicado**

A Secretaria de Saúde do Rio Grande do Sul busca identificar quais categorias de doenças apresentam maior crescimento ao longo dos anos entre a população gaúcha.

Para isso, o estudo deverá utilizar dados oficiais do DATASUS.

**Objetivos**

O estudante deverá selecionar uma única categoria da Classificação Internacional de Doenças, 10ª (CID-10) e desenvolver as seguintes análises:

* Análise de tendência temporal (4 pontos)
* Análise comparativa normalizada (2 pontos)
* Comunicação dos resultados (4 pontos)
* Produção científica (1 ponto extra)

# Carga de bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm

# Carga de dados obtidos a partir do portal DATASUS

* Morbidade Hospitalar do SUS - por local de residência - Rio Grande do Sul
* Linha: Por região e UF
* Coluna: Por ano de atendimento
* Conteúdo: Internações
* Internações por Ano atendimento segundo Região de Saúde (CIR)
* Capítulo CID-10: II. Neoplasias (tumores)
* Lista Morb CID-10: Neoplasia maligna da pele
* Período: Jan/2008-Dez/2025
* colunas separadas por ;


https://datasus.saude.gov.br/acesso-a-informacao/morbidade-hospitalar-do-sus-sih-sus/


In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/Sandrocamargo/data-science/refs/heads/main/datasets/melanoma2008-2025.txt", sep=";")
df.info()

In [ ]:
# remover última linha
total_sem_ultima = df["Total"].iloc[:-1]

plt.figure(figsize=(6, 4))
plt.boxplot(total_sem_ultima, tick_labels=["Total"])

plt.title("Boxplot da coluna Total (sem última linha)")
plt.ylabel("Casos")

plt.show()

In [ ]:
df.head(n=10)

In [ ]:
print(df.describe())

In [ ]:
df.drop(columns=["2007"], inplace=True)

In [ ]:
df.head(n=8)

In [ ]:
# --- preparar dados ---
colunas_anos = [c for c in df.columns if c.isdigit()]
anos = np.array(colunas_anos, dtype=int)

# matriz de valores
Y = df[colunas_anos].astype(float).values

# matriz X com intercepto (uma vez só)
X = sm.add_constant(anos)

# --- regressão ---
slopes = []
interceptos = []
p_values = []
r2_adj = []
medias = []

for y in Y:
    model = sm.OLS(y, X).fit()

    slopes.append(model.params[1])
    interceptos.append(model.params[0])
    p_values.append(model.pvalues[1])
    r2_adj.append(model.rsquared_adj)
    medias.append(y.mean())

# converter para array
slopes = np.array(slopes)
medias = np.array(medias)

razoes = slopes / medias

# --- dataframe final ---
resultados_df = pd.DataFrame({
    "Regiao": df.iloc[:, 0],
    "Coef_Angular": slopes,
    "Intercepto": interceptos,
    "p_valor": p_values,
    "R2_ajustado": r2_adj,
    "Media": medias,
    "Razao": razoes
}).sort_values(by="Razao", ascending=False)

resultados_df.head(n=110)

In [ ]:
# --- identificar extremos ---
regiao_max = resultados_df.iloc[0]["Regiao"]
regiao_min = resultados_df.iloc[-1]["Regiao"]

# --- função para obter série ---
def get_series(regiao):
    return df.loc[df.iloc[:, 0] == regiao, colunas_anos].values.flatten()

# --- função para tendência + coeficientes ---
def trend(anos, y):
    model = LinearRegression()
    X = np.array(anos).reshape(-1, 1)
    model.fit(X, y)

    y_pred = model.predict(X)
    a = model.coef_[0]
    b = model.intercept_
    r2 = model.score(X, y)

    return y_pred, a, b, r2

# --- dados ---
y_max = get_series(regiao_max)
y_min = get_series(regiao_min)

ymax_pred, a_max, b_max, r2_max = trend(anos, y_max)
ymin_pred, a_min, b_min, r2_min = trend(anos, y_min)

# --- plot ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

x_vals = anos.flatten()

# maior razão
axes[0].plot(x_vals, y_max, marker='o')
axes[0].plot(x_vals, ymax_pred, linestyle='--')
axes[0].set_title(f"{regiao_max}\nMaior razão")
axes[0].set_xlabel("Ano")
axes[0].set_ylabel("Casos")

eq_max = f"y = {a_max:.2f}x + {b_max:.2f}\nR² = {r2_max:.2f}"
axes[0].text(
    0.05, 0.95, eq_max,
    transform=axes[0].transAxes,
    fontsize=9,
    verticalalignment='top',
    bbox=dict(boxstyle="round", alpha=0.2)
)

# menor razão
axes[1].plot(x_vals, y_min, marker='o')
axes[1].plot(x_vals, ymin_pred, linestyle='--')
axes[1].set_title(f"{regiao_min}\nMenor razão")
axes[1].set_xlabel("Ano")

eq_min = f"y = {a_min:.2f}x + {b_min:.2f}\nR² = {r2_min:.2f}"
axes[1].text(
    0.05, 0.95, eq_min,
    transform=axes[1].transAxes,
    fontsize=9,
    verticalalignment='top',
    bbox=dict(boxstyle="round", alpha=0.2)
)

plt.tight_layout()
plt.show()

In [ ]:
import math

# --- função para obter série ---
def get_series(regiao):
    return df.loc[df.iloc[:, 0] == regiao, colunas_anos].values.flatten()

# --- função para tendência + coeficientes ---
def trend(anos, y):
    model = LinearRegression()
    X = np.array(anos).reshape(-1, 1)
    model.fit(X, y)
    y_pred = model.predict(X)

    a = model.coef_[0]
    b = model.intercept_
    r2 = model.score(X, y)  # <-- cálculo do R²

    return y_pred, a, b, r2

# --- lista de regiões ---
regioes = resultados_df["Regiao"].tolist()

# --- layout ---
n = len(regioes)
ncols = 2
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4 * nrows))
axes = axes.reshape(nrows, ncols)

# --- loop ---
for i, regiao in enumerate(regioes):
    row = i // ncols
    col = i % ncols

    ax = axes[row, col]

    y = get_series(regiao)
    y_pred, a, b, r2 = trend(anos, y)

    x_vals = anos.flatten()

    ax.plot(x_vals, y, marker='o')
    ax.plot(x_vals, y_pred, linestyle='--')

    ax.set_title(f"{regiao}")
    ax.set_xlabel("Ano")
    ax.set_ylabel("Casos")

    # --- equação + R² ---
    eq_text = f"y = {a:.2f}x + {b:.2f}\n$R^2$ = {r2:.2f}"

    ax.text(
        0.05, 0.95, eq_text,
        transform=ax.transAxes,
        fontsize=9,
        verticalalignment='top',
        bbox=dict(boxstyle="round", alpha=0.2)
    )

# --- remover eixos vazios ---
for j in range(i + 1, nrows * ncols):
    fig.delaxes(axes.flatten()[j])

plt.tight_layout()
plt.show()